# Macro EDA — FRED series inspection

Exploratory analysis of the five monthly FRED series used by the study, from the local cache. This notebook is *documentation of the data*: every modeling decision (transforms, stationarity handling, splits) is made in the pipeline and unit-tested; here we simply look at what the data look like.

**Run it after `python scripts/download_data.py`.**

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve() / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from macro_forecasting_lab.data import load_series, transform_series
from macro_forecasting_lab.config import DEFAULT_CONFIG

## 1. Load raw levels from the cache

In [ ]:
ids = ["CPIAUCSL", "UNRATE", "INDPRO", "FEDFUNDS", "M2SL"]
levels = {i: load_series(i, raw_dir=DEFAULT_CONFIG.raw_dir) for i in ids}

df = pd.DataFrame(levels)
print(df.head(3))
print("\ncoverage:", df.index.min().date(), "->", df.index.max().date(), f"({len(df)} months)")

## 2. Transformed modeling series

The two targets are CPI inflation (YoY log-difference, %) and the unemployment rate (level).

In [ ]:
infl = transform_series(levels["CPIAUCSL"], "yoy_log_diff_pct")
unrate = transform_series(levels["UNRATE"], "level")

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
infl.plot(ax=ax[0], color="tab:red"); ax[0].set_title("US CPI inflation, YoY log-diff %")
unrate.plot(ax=ax[1], color="tab:blue"); ax[1].set_title("US unemployment rate, %")
ax[1].axvline(pd.Timestamp("2016-01-01"), color="k", ls="--", lw=1, label="test start")
ax[1].legend()
fig.tight_layout(); plt.show()

## 3. Train/test shading and the COVID break

The evaluation window (2016+) contains the 2020 COVID disruption — visible as the spikes in both panels. The study reports metrics with and without 2020-02 → 2020-12 for exactly this reason.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
infl.loc["2014":].plot(ax=ax, color="tab:red")
ax.axvspan(pd.Timestamp("2020-02-01"), pd.Timestamp("2020-12-31"), color="grey", alpha=0.3, label="excluded COVID window")
ax.set_title("Inflation 2014–2026 with the structural-break window shaded")
ax.legend(); fig.tight_layout(); plt.show()

## Takeaways

- Inflation is stationary as a YoY log-difference (d=1 chosen for levels); unemployment is stationary in levels.
- The 2020 shock dominates any 2016+ evaluation of unemployment; naive persistence is nearly unbeatable outside it.
- All decisions above are encoded in `configs/default.yaml` and exercised by `scripts/run_experiments.py`.